<a href="https://colab.research.google.com/github/Bendy545/CS2-Match-Predictor/blob/dev/cs2_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
import numpy as np
import json
import pickle

df = pd.read_csv('faceit_clean.csv')

separating features  
separating output for classification model and regression model

In [9]:
label_cols = ['team1_win', 'score_diff']
cat_cols = ['map']
num_cols = [c for c in df.columns if c not in label_cols + cat_cols]

X = df[num_cols + cat_cols]
y_cls = df['team1_win']
y_reg = df['score_diff']

# Train / Test split

In [10]:
from sklearn.model_selection import train_test_split, cross_val_score

In [11]:
X_train, X_test, y_cls_train, y_cls_test, y_reg_train, y_reg_test = train_test_split(
    X, y_cls, y_reg,
    test_size=0.2,
    random_state=42,
    stratify=y_cls
)

# Preprocessing Pipeline

In [12]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols),
    ]
)

In [14]:
baseline_acc = max(y_cls.mean(), 1 - y_cls.mean())

LogisticRegression model

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score
)

log_reg_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

log_reg_pipe.fit(X_train, y_cls_train)

y_pred_lr = log_reg_pipe.predict(X_test)
y_proba_lr = log_reg_pipe.predict_proba(X_test)[:, 1]

cv_lr = cross_val_score(log_reg_pipe, X_train, y_cls_train, cv=5, scoring='accuracy')

print(accuracy_score(y_cls_test, y_pred_lr))

0.6866666666666666
